# Spark SQL Catalyst: How a Query Becomes a Plan

Run this notebook cell-by-cell against the live cluster spawned from the topic page. See `concept.md` for the what/why background.

The same logical query -- join two tables, then filter on a single-side numeric column -- is expressed three ways below: DataFrame API, raw SQL, and a Python-UDF-wrapped condition. For each cell: **form a hypothesis before running** (will the `Filter` sit above or below the `SortMergeJoin`?), run it, read `.explain(mode="formatted")` yourself, *then* call `playbook.checkpoint(df, topic="catalyst-plans")` and click **Reveal self-check** on the topic page to compare against the manifest-driven annotation (US-SH8).

In [ ]:
import sys
sys.path.insert(0, "/workspace")

from pyspark.sql import Row, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import BooleanType
from driver.playbook import checkpoint

spark = SparkSession.builder.appName("catalyst-plans").getOrCreate()

# Disable broadcast joins entirely so all three variants below deterministically pick the same
# join strategy (SortMergeJoin) -- this topic is about filter *position* relative to the join, not
# about which join strategy gets picked (that's join-strategies' topic), so pinning it removes a
# variable that would otherwise make the three plans harder to compare apples-to-apples.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")
print("Spark version:", spark.version)

## Setup: two tables, joined on `customer_id`

`orders` (the larger side) and `customers` (the dimension side), joined on `customer_id`. All three variants below apply the identical condition -- `orders.amount > THRESHOLD` -- which references only the `orders` side, so Catalyst *can* legally push it below the join in every variant. Whether it actually does is exactly what each cell's plan will show.

In [ ]:
SCRATCH_DIR = "/workspace/scratch/catalyst-plans"
ORDERS_ROWS = 500_000
CUSTOMERS_ROWS = 50_000
THRESHOLD = 500.0

customers_rdd = spark.sparkContext.parallelize(range(CUSTOMERS_ROWS), 8).map(
    lambda i: Row(customer_id=i, region=f"region-{i % 10}")
)
customers_df = spark.createDataFrame(customers_rdd)

orders_rdd = spark.sparkContext.parallelize(range(ORDERS_ROWS), 16).map(
    lambda i: Row(order_id=i, customer_id=i % CUSTOMERS_ROWS, amount=float(i % 1000))
)
orders_df = spark.createDataFrame(orders_rdd)

# Round-trip through parquet so both sides carry real file-size stats and a realistic Scan node,
# matching join-strategies' convention -- a createDataFrame()-backed DataFrame has no size stats
# Catalyst can use, which would otherwise make Scan nodes look different from a real table read.
customers_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/customers")
orders_df.write.mode("overwrite").parquet(f"{SCRATCH_DIR}/orders")
customers_df = spark.read.parquet(f"{SCRATCH_DIR}/customers")
orders_df = spark.read.parquet(f"{SCRATCH_DIR}/orders")

print("orders rows:", orders_df.count(), " customers rows:", customers_df.count())

## 1. DataFrame API

`orders_df.join(customers_df, on="customer_id").filter(F.col("amount") > THRESHOLD)`.

**Hypothesis:** does the `Filter` node appear above or below `SortMergeJoin` in the plan?

Note: because the join key (`customer_id`) is nullable, the analyzer also inserts its own `isnotnull(customer_id)` null-check filter on each side of the join, and that filter gets pushed down too -- so it is expected to see *two* nodes labeled pushed-down-below-join here, not one. One is this exercise's `amount > THRESHOLD` filter; the other is Spark's own join-key null-check, not a duplicate or a bug.

In [ ]:
dataframe_filtered = orders_df.join(customers_df, on="customer_id", how="inner") \
    .filter(F.col("amount") > THRESHOLD)
dataframe_filtered.count()
dataframe_filtered.explain(mode="formatted")

In [ ]:
checkpoint(dataframe_filtered, topic="catalyst-plans")
print("Checkpoint written -- click 'Reveal self-check' on the topic page.")

## 2. Raw SQL -- the same query, a different surface syntax

Same join, same condition, expressed as a `WHERE` clause over temp views of the *same two DataFrames*. By the time this reaches the analyzer, there is nothing left to distinguish it from cell 1's DataFrame version -- both compile to the same logical plan (see `concept.md`).

**Hypothesis:** will this plan differ from cell 1's in any way that matters?

Note: as in cell 1, expect two pushed-down-below-join filters here, not one -- the `amount > THRESHOLD` filter and Spark's automatic `isnotnull(customer_id)` join-key null-check (see cell 1's note).

In [ ]:
orders_df.createOrReplaceTempView("orders")
customers_df.createOrReplaceTempView("customers")

sql_filtered = spark.sql(f"""
    SELECT o.order_id, o.customer_id, o.amount, c.region
    FROM orders o
    JOIN customers c ON o.customer_id = c.customer_id
    WHERE o.amount > {THRESHOLD}
""")
sql_filtered.count()
sql_filtered.explain(mode="formatted")

In [ ]:
checkpoint(sql_filtered, topic="catalyst-plans")
print("Checkpoint written -- Reveal again and compare against cell 1's plan: same shape.")

## 3. UDF-wrapped filter -- the same condition, made opaque to Catalyst

Identical condition (`amount > THRESHOLD`), now wrapped in a Python UDF marked `.asNondeterministic()`. Catalyst will not reorder a non-deterministic expression past a join -- the same conservative treatment any genuinely opaque UDF gets, since Catalyst has no way to look inside Python bytecode to prove it's safe to move.

**Hypothesis:** does the `Filter` still sit below the join here, or has it moved?

In [ ]:
is_high_value = F.udf(lambda amount: amount > THRESHOLD, BooleanType()).asNondeterministic()

udf_filtered = orders_df.join(customers_df, on="customer_id", how="inner") \
    .filter(is_high_value(F.col("amount")))
udf_filtered.count()
udf_filtered.explain(mode="formatted")

In [ ]:
checkpoint(udf_filtered, topic="catalyst-plans")
print("Checkpoint written -- Reveal again. The Filter here should be labeled distinctly from cells "
      "1 and 2 (US-SH8): still pending above the join, next to the BatchEvalPython node evaluating "
      "the UDF, instead of pushed down onto the orders scan.")

## 4. Reset session-level confs

Reset the confs this notebook changed so a later run (or another topic reusing this session) isn't affected.

In [ ]:
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", 10 * 1024 * 1024)